In [ ]:
# --------------------------------------------------------------
# 0. Install required packages (run once)
# --------------------------------------------------------------
!pip install -q ultralytics matplotlib ipywidgets

# --------------------------------------------------------------
# 1. Imports & environment detection
# --------------------------------------------------------------
import sys, time, io, os
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ---- Colab-specific imports (only executed in Colab) ----
if 'google.colab' in sys.modules:
    from google.colab import files
    from google.colab.patches import cv2_imshow
else:
    import ipywidgets as widgets
    from IPython.display import display, Image as IPyImage

from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator

# --------------------------------------------------------------
# 2. Helper functions
# --------------------------------------------------------------
def upload_file():
    """Return the path to the uploaded video file."""
    if 'google.colab' in sys.modules:
        print("Please upload your video file.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        video_filename = next(iter(uploaded))
        tmp_path = f"/tmp/{video_filename}"
        with open(tmp_path, "wb") as f:
            f.write(uploaded[video_filename])
        return tmp_path

    else:   # non-Colab Jupyter
        uploader = widgets.FileUpload(accept='video/*', multiple=False)
        display(uploader)
        print("Please upload a video file using the widget above.")
        while not uploader.value:
            time.sleep(0.5)
        fname = list(uploader.value.keys())[0]
        tmp_path = os.path.abspath(fname)
        with open(tmp_path, "wb") as f:
            f.write(uploader.value[fname]["content"])
        return tmp_path

def show_frame(frame):
    """Display a frame – cv2_imshow in Colab, matplotlib elsewhere."""
    if 'google.colab' in sys.modules:
        cv2_imshow(frame)
    else:
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(img_rgb)
        plt.axis('off')
        plt.show()

# --------------------------------------------------------------
# 3. Load model & initialise traffic-light state
# --------------------------------------------------------------
model = YOLO('yolov8m.pt')               # change to yolov8n.pt for speed
targetClasses = [0, 2, 15, 16]           # person, car, cat, dog

# Traffic-light timers
current_light_state = "Green"
state_start_time = time.time()
green_light_duration = 10
yellow_light_duration = 90
red_light_duration = 15

# Motion tracking
previous_frame_car_boxes = []
movement_threshold = 10

# Red-light “no-person/pet” counter
no_detection_red_count = 0

# --------------------------------------------------------------
# 4. Upload video
# --------------------------------------------------------------
video_path = upload_file()
print(f"Uploaded video saved to: {video_path}")

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {video_path}")

# Video writer setup
frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30

output_video_path = 'detection-output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

frame_count = 0

# --------------------------------------------------------------
# 5. Main processing loop
# --------------------------------------------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    # Process every Xth frame (feel free to change)
    if frame_count % 1 != 0:
        out.write(frame)                 # keep original frames in output
        continue

    print(f"Processing frame {frame_count}...")
    results = model(frame)[0]
    annotator = Annotator(frame.copy())

    detected_objects_in_frame = []
    is_moving_car_detected = False
    is_person_cat_or_dog_detected = False
    current_frame_car_boxes = []

    # ------------------------------------------------------------------
    # 5.1  Iterate over detections
    # ------------------------------------------------------------------
    for box in results.boxes:
        cls = int(box.cls)
        if cls not in targetClasses:
            continue

        b = box.xyxy[0].cpu().numpy()
        label = model.names[cls]
        annotator.box_label(b, label)
        detected_objects_in_frame.append(label)

        if cls == 2:                     # car
            c_x = (b[0] + b[2]) / 2
            c_y = (b[1] + b[3]) / 2
            # motion check against previous frame
            for _, prev_x, prev_y in previous_frame_car_boxes:
                dist = np.hypot(c_x - prev_x, c_y - prev_y)
                if dist > movement_threshold:
                    is_moving_car_detected = True
                    break
            current_frame_car_boxes.append((b, c_x, c_y))

        if cls in (0, 15, 16):           # person / cat / dog
            is_person_cat_or_dog_detected = True

    previous_frame_car_boxes = current_frame_car_boxes

    # ------------------------------------------------------------------
    # 5.2  Traffic-light state machine
    # ------------------------------------------------------------------
    elapsed = time.time() - state_start_time
    trigger_message = ""

    # Red-light “no-person/pet” counter
    if current_light_state == "Red":
        if is_person_cat_or_dog_detected:
            no_detection_red_count = 0
        else:
            no_detection_red_count += 1
    else:
        no_detection_red_count = 0

    if current_light_state == "Green":
        if elapsed >= green_light_duration:
            if is_moving_car_detected and is_person_cat_or_dog_detected:
                trigger_message = " - Moving Car & Person/Pet Detected"
                current_light_state = "Yellow"
                state_start_time = time.time()
            elif is_moving_car_detected:
                trigger_message = " - Moving Car Detected"
            elif is_person_cat_or_dog_detected:
                trigger_message = " - Person/Pet Detected"

    elif current_light_state == "Yellow":
        if elapsed >= yellow_light_duration:
            current_light_state = "Red"
            state_start_time = time.time()

    elif current_light_state == "Red":
        if elapsed >= red_light_duration and no_detection_red_count >= 2:
            current_light_state = "Green"
            state_start_time = time.time()

    # ------------------------------------------------------------------
    # 5.3  Draw traffic-light UI on the frame
    # ------------------------------------------------------------------
    annotated = annotator.result()

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    thickness = 2
    padding = 10

    display_text = f"{current_light_state} Light"
    (tw, th), baseline = cv2.getTextSize(display_text, font, font_scale, thickness)

    # background rectangle
    cv2.rectangle(annotated,
                  (padding, padding),
                  (padding + tw + 10, padding + th + baseline + 10),
                  (0, 0, 0), -1)

    # text colour
    colour_map = {"Green": (0,255,0), "Yellow": (0,255,255), "Red": (0,0,255)}
    txt_col = colour_map[current_light_state]
    cv2.putText(annotated, display_text,
                (padding, padding + th),
                font, font_scale, txt_col, thickness, cv2.LINE_AA)

    # traffic-light circle (top-right)
    h, w = annotated.shape[:2]
    circle_radius = 20
    circle_center = (w - padding - circle_radius, padding + circle_radius)
    cv2.circle(annotated, circle_center, circle_radius, txt_col, -1)
    cv2.circle(annotated, circle_center, circle_radius, (255,255,255), 2)

    # ------------------------------------------------------------------
    # 5.4  Console output & frame display
    # ------------------------------------------------------------------
    if detected_objects_in_frame or current_light_state != "Green":
        objs = ', '.join(set(detected_objects_in_frame))
        print(f"Detected in frame {frame_count}: {objs}")
        if is_moving_car_detected:
            print(f" -> Moving car detected!")
        if is_person_cat_or_dog_detected:
            print(f" -> Person/Cat/Dog detected!")
        print(f" -> Streetlight: {display_text}{trigger_message}")
        show_frame(annotated)

    out.write(annotated)

# --------------------------------------------------------------
# 6. Cleanup
# --------------------------------------------------------------
cap.release()
out.release()
print("\nVideo processing complete.")
print(f"Output saved as: {output_video_path}")

# In non-Colab notebooks show the saved video
if 'google.colab' not in sys.modules:
    display(IPyImage(output_video_path))